<a href="https://colab.research.google.com/github/Chanthul4054/E-Policing-An-Integrated-Spatio-Temporal-Crime-Forecasting-and-Decision-Support-System/blob/Pattern-Recognition-System/rule_filtering_and_pattern_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [28]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules

DATASET_PATH = "/content/pattern_recognition_preprocessed.csv"

In [29]:
TOP_K_PATTERNS = 10

MIN_SUPPORT = 0.03
MIN_CONFIDENCE = 0.30
MIN_LIFT = 1.05

MAX_RULE_LEN = 4

MIN_CONTEXT_ITEMS = 2
CONF_SIMILARITY_TOL = 0.02

OUTPUT_JSON = "final_rule_database.json"

In [30]:
df = pd.read_csv(DATASET_PATH)
print("Dataset shape:", df.shape)

required_cols = ["gn_pcode", "gn_division"]

for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Dataset missing required column: {c}")

Dataset shape: (1608, 46)


In [31]:
# AUTO DETECT FEATURE COLUMNS

crime_columns = [c for c in df.columns if c.startswith("crime_")]
location_columns = [c for c in df.columns if c.startswith("loc_")]

excluded_cols = set(["gn_pcode", "gn_division"])
excluded_cols |= set(crime_columns)
excluded_cols |= set(location_columns)

context_columns = []

In [32]:
for c in df.columns:
    if c in excluded_cols:
        continue

    if df[c].dropna().isin([0, 1, True, False]).all():
        context_columns.append(c)

selected_columns = context_columns + location_columns + crime_columns

In [33]:
print("Detected crime columns:", len(crime_columns))
print("Detected location columns:", len(location_columns))
print("Detected context columns:", len(context_columns))

Detected crime columns: 6
Detected location columns: 13
Detected context columns: 13


In [34]:
# HELPER FUNCTIONS

def context_to_crime(rule_row):
    antecedents = rule_row["antecedents"]
    consequents = rule_row["consequents"]

    ant_has_crime = any(item.startswith("crime_") for item in antecedents)
    con_has_crime = any(item.startswith("crime_") for item in consequents)

    return (not ant_has_crime) and con_has_crime

In [35]:
def get_rule_context_items(rule_row):
    ants = set(rule_row["antecedents"])
    context_items = [x for x in ants if (not x.startswith("crime_")) and (not x.startswith("loc_"))]
    return context_items

In [36]:
def contains_required_structure(rule_row):

    ctx = get_rule_context_items(rule_row)

    has_time = any(("time_" in x) for x in ctx)
    has_day = any(("day_" in x) or ("weekday" in x) or ("weekend" in x) for x in ctx)

    return has_time and has_day

In [37]:
def subset_prune_rules(rules_df, conf_tol=0.02):

    rules_df = rules_df.copy()

    keep = [True] * len(rules_df)
    rules_list = rules_df.to_dict("records")

    for i in range(len(rules_list)):

        if not keep[i]:
            continue

        A = rules_list[i]
        A_ant = set(A["antecedents"])
        A_conf = float(A["confidence"])

        for j in range(len(rules_list)):

            if i == j or not keep[j]:
                continue

            B = rules_list[j]
            B_ant = set(B["antecedents"])
            B_conf = float(B["confidence"])

            if A_ant.issubset(B_ant) and len(B_ant) > len(A_ant):

                if abs(A_conf - B_conf) <= conf_tol:
                    keep[j] = False

    return rules_df.loc[keep].copy()

In [39]:
# MAIN RULE GENERATION LOOP
ALL_RULES = []

unique_gns = df[["gn_pcode", "gn_division"]].drop_duplicates().reset_index(drop=True)

for _, gn_row in unique_gns.iterrows():

    GN_PCODE = gn_row["gn_pcode"]
    GN_DIV = gn_row["gn_division"]

    df_gn = df[df["gn_pcode"] == GN_PCODE].copy()

    if df_gn.empty:
        continue

    for loc_col in location_columns:

        df_loc = df_gn[df_gn[loc_col] == 1].copy()

        if df_loc.empty:
            continue

        transaction_df = df_loc[selected_columns].fillna(0).astype(int)
        transaction_df = transaction_df.map(lambda x: True if x == 1 else False)

        frequent_itemsets = apriori(
            transaction_df,
            min_support=MIN_SUPPORT,
            use_colnames=True,
            max_len=MAX_RULE_LEN
        )